# Deep Stratified Cox Tutorial

This notebook is an expanded version of `tutorial_deep_stratified_cox.py`.

The script runs the whole workflow end to end. The notebook keeps the same core workflow, but pauses at the useful intermediate objects:

- the simulated survival data,
- the train/test split and standardized covariates,
- the stratum labels,
- one event-centered same-stratum risk set,
- one DataLoader batch of variable-length risk-set components,
- the component Cox loss,
- model fitting and held-out C-index evaluation.

The model is a neural network risk score plus a stratified Cox partial likelihood. There is no KL distillation and no teacher model here.

## 1. Imports

The high-level imports mirror the tutorial script. The lower-level imports expose the dataset and loss functions so we can inspect the risk-set construction directly.

In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from nnstratcox import (
    MLPRisk,
    concordance_index,
    fit_model,
    predict_risk,
    simulate_centered_survival,
    standardize_train_test,
    train_test_split,
)
from nnstratcox.dataset import StratifiedRiskSetDataset, risk_set_collate
from nnstratcox.loss import cox_component_loss, stratified_cox_loss

## 2. Configuration

For interactive use, the default notebook values are a little smaller than a full run. To match the script more closely, use `N = 600`, `EPOCHS = 100`, and `BATCH_SIZE = 16`.

In [2]:
SEED = 123
N = 300
N_CENTERS = 8
EPOCHS = 30
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE

'cpu'

## 3. Simulate Center-Stratified Survival Data

`simulate_centered_survival` creates data where the covariate effect is shared across centers, but each center has its own baseline hazard. This is exactly the setting where stratified Cox is useful: compare patients within center while learning one shared neural network risk function.

In [3]:
data = simulate_centered_survival(n=N, n_centers=N_CENTERS, seed=SEED)

print("x shape:", data.x.shape)
print("duration shape:", data.duration.shape)
print("event shape:", data.event.shape)
print("center shape:", data.center.shape)
print("overall event rate:", round(float(data.event.mean()), 3))
print("centers:", np.unique(data.center))

x shape: (300, 8)
duration shape: (300,)
event shape: (300,)
center shape: (300,)
overall event rate: 0.433
centers: [0 1 2 3 4 5 6 7]


A quick center-level summary helps verify that `center` is the intended stratum variable.

In [4]:
def center_summary(center, event, duration):
    rows = []
    for c in np.unique(center):
        idx = center == c
        rows.append(
            {
                "center": int(c),
                "n": int(idx.sum()),
                "events": int(event[idx].sum()),
                "event_rate": round(float(event[idx].mean()), 3),
                "median_duration": round(float(np.median(duration[idx])), 3),
            }
        )
    return rows

center_summary(data.center, data.event, data.duration)

[{'center': 0,
  'n': 41,
  'events': 13,
  'event_rate': 0.317,
  'median_duration': 22.372},
 {'center': 1,
  'n': 36,
  'events': 8,
  'event_rate': 0.222,
  'median_duration': 25.145},
 {'center': 2,
  'n': 37,
  'events': 21,
  'event_rate': 0.568,
  'median_duration': 11.497},
 {'center': 3,
  'n': 35,
  'events': 32,
  'event_rate': 0.914,
  'median_duration': 8.879},
 {'center': 4,
  'n': 36,
  'events': 14,
  'event_rate': 0.389,
  'median_duration': 15.002},
 {'center': 5,
  'n': 40,
  'events': 19,
  'event_rate': 0.475,
  'median_duration': 20.202},
 {'center': 6,
  'n': 44,
  'events': 15,
  'event_rate': 0.341,
  'median_duration': 11.788},
 {'center': 7,
  'n': 31,
  'events': 8,
  'event_rate': 0.258,
  'median_duration': 41.958}]

## 4. Train/Test Split and Standardization

The split is the same as in the script. Covariates are standardized using the training-set mean and standard deviation, then the same transformation is applied to the test set.

In [5]:
train, test = train_test_split(data, test_size=0.25, seed=SEED + 1)
x_train, x_test = standardize_train_test(train.x, test.x)

print("train n:", x_train.shape[0])
print("test n:", x_test.shape[0])
print("train event rate:", round(float(train.event.mean()), 3))
print("test event rate:", round(float(test.event.mean()), 3))
print("standardized training means, first 5:", np.round(x_train.mean(axis=0)[:5], 4))
print("standardized training stds, first 5:", np.round(x_train.std(axis=0)[:5], 4))

train n: 225
test n: 75
train event rate: 0.453
test event rate: 0.373
standardized training means, first 5: [-0. -0. -0.  0.  0.]
standardized training stds, first 5: [1. 1. 1. 1. 1.]


## 5. What Stratification Means

The network outputs one scalar risk score for each patient:

```text
h_i = f_theta(X_i)
```

For an event case `i`, the stratified Cox risk set is restricted to the same stratum:

```text
R_i = {j: duration_j >= duration_i and strata_j == strata_i}
loss_i = -h_i + log sum_{j in R_i} exp(h_j)
```

If `strata` is hospital center, each event case is compared only with patients still at risk in the same hospital center. The neural network parameters are still shared across all centers.

## 6. Build Event-Centered Same-Stratum Risk Sets

`StratifiedRiskSetDataset` creates one dataset item per observed event. Each item contains:

```text
[event case i] + [all j with duration_j >= duration_i and strata_j == strata_i]
```

Inside that component, the event case has local event label 1 and everyone else has local event label 0. This local relabeling is why each training component has exactly one event.

In [6]:
risk_dataset = StratifiedRiskSetDataset(
    x_train,
    train.duration,
    train.event,
    train.center,
    max_controls=None,
    seed=SEED,
)

print("number of observed training events:", int(train.event.sum()))
print("number of dataset components:", len(risk_dataset))

number of observed training events: 102
number of dataset components: 102


Inspect one component. We reconstruct the original risk-set indices so it is clear that all controls are from the same center and are still at risk at the event time.

In [7]:
component_id = 0
case_idx = int(risk_dataset.case_indices[component_id])
same_center = train.center == train.center[case_idx]
at_risk = train.duration >= train.duration[case_idx]
risk_idx = np.where(same_center & at_risk)[0]

x_comp, duration_comp, event_comp = risk_dataset[component_id]

print("case original index:", case_idx)
print("case center:", int(train.center[case_idx]))
print("case duration:", round(float(train.duration[case_idx]), 3))
print("case observed event:", int(train.event[case_idx]))
print("risk set size:", len(risk_idx))
print("component tensor size:", len(duration_comp))
print("local event count in component:", int(event_comp.sum().item()))
print("all same center:", bool(np.all(train.center[risk_idx] == train.center[case_idx])))
print("all at risk at case time:", bool(np.all(train.duration[risk_idx] >= train.duration[case_idx])))

case original index: 0
case center: 3
case duration: 13.25
case observed event: 1
risk set size: 11
component tensor size: 11
local event count in component: 1
all same center: True
all at risk at case time: True


The component is sorted by duration so the loss can use suffix risk-set sums. The row with `local_event = 1` is the event case; the other rows are controls for this local Cox component.

In [8]:
preview_n = min(10, len(duration_comp))
component_preview = [
    {
        "local_row": k,
        "duration": round(float(duration_comp[k]), 3),
        "local_event": int(event_comp[k]),
    }
    for k in range(preview_n)
]
component_preview

[{'local_row': 0, 'duration': 13.25, 'local_event': 1},
 {'local_row': 1, 'duration': 14.335, 'local_event': 0},
 {'local_row': 2, 'duration': 15.431, 'local_event': 0},
 {'local_row': 3, 'duration': 18.125, 'local_event': 0},
 {'local_row': 4, 'duration': 20.208, 'local_event': 0},
 {'local_row': 5, 'duration': 29.213, 'local_event': 0},
 {'local_row': 6, 'duration': 33.181, 'local_event': 0},
 {'local_row': 7, 'duration': 35.065, 'local_event': 0},
 {'local_row': 8, 'duration': 39.297, 'local_event': 0},
 {'local_row': 9, 'duration': 54.679, 'local_event': 0}]

## 7. Inspect One DataLoader Batch

The DataLoader does not stack patients into a rectangular tensor. Each event case can have a different risk-set size, so the custom collate function returns lists of tensors.

`BATCH_SIZE` means the number of event-centered Cox components per optimizer step, not the number of patients.

In [9]:
loader = DataLoader(
    risk_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=risk_set_collate,
)

xs_list, duration_list, event_list = next(iter(loader))

print("number of components in this batch:", len(xs_list))
print("component sizes:", [len(t) for t in duration_list])
print("local event counts:", [int(e.sum().item()) for e in event_list])
print("total patients processed in this optimizer step:", sum(len(t) for t in duration_list))

number of components in this batch: 8
component sizes: [14, 20, 30, 16, 4, 29, 12, 13]
local event counts: [1, 1, 1, 1, 1, 1, 1, 1]
total patients processed in this optimizer step: 138


## 8. Inspect One Component Loss

For the current event-centered construction, one component has one local event. The component Cox loss is:

```text
-h_case + log sum_{j in component} exp(h_j)
```

This is the one-event Cox partial likelihood term for that event case's same-stratum risk set.

In [10]:
demo_model = MLPRisk(input_dim=x_train.shape[1], hidden_dim=64, num_layers=2, dropout=0.10, seed=SEED).to(DEVICE)

x0 = xs_list[0].to(DEVICE)
t0 = duration_list[0].to(DEVICE)
e0 = event_list[0].to(DEVICE)

scores0 = demo_model(x0)
loss0 = cox_component_loss(scores0, t0, e0)

case_pos = int(torch.where(e0 == 1)[0][0].item())
manual_loss0 = -scores0[case_pos] + torch.logsumexp(scores0, dim=0)

print("component size:", len(t0))
print("case local row:", case_pos)
print("cox_component_loss:", float(loss0.detach().cpu()))
print("manual one-event loss:", float(manual_loss0.detach().cpu()))

component size: 14
case local row: 0
cox_component_loss: 3.308901309967041
manual one-event loss: 3.308901309967041


## 9. Full Stratified Loss on the Validation/Test Set

`stratified_cox_loss` is useful for validation. It loops over strata, computes the Cox partial likelihood inside each stratum, sums across strata, and divides by the total number of events.

This full-data version also handles tied event times with the Breslow tied-time form.

In [11]:
x_te = torch.as_tensor(x_test, dtype=torch.float32, device=DEVICE)
t_te = torch.as_tensor(test.duration, dtype=torch.float32, device=DEVICE)
e_te = torch.as_tensor(test.event, dtype=torch.float32, device=DEVICE)
s_te = torch.as_tensor(test.center, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    initial_test_loss = stratified_cox_loss(demo_model(x_te), t_te, e_te, s_te)

float(initial_test_loss.detach().cpu())

1.9392746686935425

## 10. Train the Deep Stratified Cox Model

`fit_model` uses the event-centered DataLoader internally. Each optimizer step averages the losses from several risk-set components:

```text
batch_loss = mean(component_loss_1, ..., component_loss_B)
```

With `max_controls=None`, every component uses the full same-stratum risk set.

In [12]:
strat_model = MLPRisk(input_dim=x_train.shape[1], hidden_dim=64, num_layers=2, dropout=0.10, seed=SEED)

history = fit_model(
    strat_model,
    x_train,
    train.duration,
    train.event,
    train.center,
    x_val=x_test,
    duration_val=test.duration,
    event_val=test.event,
    strata_val=test.center,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=EPOCHS,
    patience=40,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

print("best epoch:", history.best_epoch + 1)
print("last train loss:", round(history.train_loss[-1], 4))
print("last validation loss:", round(history.val_loss[-1], 4))

/home/dengfy/anaconda3/envs/syn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


epoch=0001 train_loss=2.6770 val_loss=1.4222
best epoch: 4
last train loss: 1.4448
last validation loss: 1.8404


## 11. Train an Ordinary Cox Neural Net Baseline

The ordinary Cox baseline uses the same network and training loop, but assigns every patient to the same stratum. This removes center-specific baseline hazards from the Cox likelihood.

In [13]:
ordinary_model = MLPRisk(input_dim=x_train.shape[1], hidden_dim=64, num_layers=2, dropout=0.10, seed=SEED + 10)

ordinary_history = fit_model(
    ordinary_model,
    x_train,
    train.duration,
    train.event,
    np.zeros_like(train.center),
    x_val=x_test,
    duration_val=test.duration,
    event_val=test.event,
    strata_val=np.zeros_like(test.center),
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=EPOCHS,
    patience=40,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    verbose=False,
)

print("best epoch:", ordinary_history.best_epoch + 1)
print("last train loss:", round(ordinary_history.train_loss[-1], 4))
print("last validation loss:", round(ordinary_history.val_loss[-1], 4))

best epoch: 2
last train loss: 3.5082
last validation loss: 4.2506


## 12. Held-Out Evaluation

The ordinary C-index compares across all patients. The stratified C-index only compares patients within the same center, matching the stratified Cox target.

In [14]:
risk_strat = predict_risk(strat_model, x_test, device=DEVICE)
risk_ordinary = predict_risk(ordinary_model, x_test, device=DEVICE)

results = {
    "stratified_model_ordinary_cindex": round(float(concordance_index(test.event, test.duration, risk_strat)), 3),
    "stratified_model_stratified_cindex": round(float(concordance_index(test.event, test.duration, risk_strat, test.center)), 3),
    "ordinary_model_ordinary_cindex": round(float(concordance_index(test.event, test.duration, risk_ordinary)), 3),
    "ordinary_model_stratified_cindex": round(float(concordance_index(test.event, test.duration, risk_ordinary, test.center)), 3),
}

results

{'stratified_model_ordinary_cindex': 0.792,
 'stratified_model_stratified_cindex': 0.815,
 'ordinary_model_ordinary_cindex': 0.778,
 'ordinary_model_stratified_cindex': 0.795}

## 13. Using Your Own Data

Replace the simulated arrays with your data:

```python
X        # shape (n, p), numeric covariates
duration # shape (n,), observed follow-up time
event    # shape (n,), 1 if event occurred, 0 if censored
strata   # shape (n,), center/site/matched-group labels
```

If the raw stratum labels are strings, encode them as integer labels before passing them into `fit_model`.

The key modeling choice is the definition of `strata`. For hospital-center stratification, `strata` should be the hospital or transplant-center label. Then every event is compared only against patients at risk within the same center.